In [ ]:


# %% [code]
import os, sys, random
from typing import Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# 如需：把 guided-diffusion 加到 PYTHONPATH
# sys.path.append("./guided-diffusion")
from guided_diffusion import dist_util
from guided_diffusion.script_util import (
    model_and_diffusion_defaults,
    create_model_and_diffusion,
)

# %% [code]
# ==============================
# Config
# ==============================
CONFIG = {
    "TRAIN_ROOT": "/path/to/train512",    # 512x512 RGB 训练集
    "VAL_ROOT":   "/path/to/val512",      # 512x512 RGB 验证集
    "CKPT_PATH":  "/path/to/512x512_model.pt",  # 预训练 guided-diffusion 512x512 权重
    "OUTDIR":     "./checkpoints_pixel512",
    "BATCH":      4,
    "LR":         2e-5,
    "EPOCHS":     200,
    "PATIENCE":   10,
    "CLASS_COND": False,     # 若使用类条件权重，设为 True
    "IMAGE_SIZE": 512,
    "NUM_WORKERS": 2,
    "SAMPLE_EVERY": 5,       # 每多少 epoch 采样
    "SAMPLE_STEPS": 250,     # 采样步数（快速预览）
}

os.makedirs(CONFIG["OUTDIR"], exist_ok=True)
# guided-diffusion 分布式环境初始化（单卡也可）
dist_util.setup_dist()
device = dist_util.dev()
print("Device:", device)

# %% [code]
# 🔽 Download pretrained 512×512 weights
# 两种可用的像素域权重：
# 1) OpenAI Guided-Diffusion 512×512（类条件）
# 2) Hugging Face 上的无条件 512×512（从 OpenAI 512 微调而来）

import requests
from pathlib import Path

USE_OPENAI_512 = True   # 下载 OpenAI 官方 512×512（类条件）
USE_HF_UNCOND = False   # 或下载 HF 无条件 512×512（lowlevelware）

MODELS_DIR = Path("./models"); MODELS_DIR.mkdir(exist_ok=True, parents=True)


def download_file(url: str, dst: Path):
    """Stream download with a simple progress indicator."""
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0))
        downloaded = 0
        with open(dst, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total:
                        pct = downloaded / total * 100
                        print(f"\rDownloading {dst.name}: {pct:5.1f}%", end="")
        print()

resolved_ckpt = None

if USE_OPENAI_512:
    OPENAI_512_URL = "https://openaipublic.blob.core.windows.net/diffusion/jul-2021/512x512_diffusion.pt"
    dst = MODELS_DIR / "512x512_diffusion.pt"
    if not dst.exists():
        print(f"Downloading OpenAI 512x512 diffusion to {dst} ...")
        download_file(OPENAI_512_URL, dst)
    else:
        print(f"Already exists: {dst}")
    resolved_ckpt = str(dst)

if USE_HF_UNCOND:
    try:
        from huggingface_hub import snapshot_download
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
        from huggingface_hub import snapshot_download

    repo_id = "lowlevelware/512x512_diffusion_unconditional_ImageNet"
    cache_dir = MODELS_DIR / "hf_uncond_512"
    print(f"Downloading snapshot of {repo_id} ...")
    local_dir = snapshot_download(repo_id=repo_id, local_dir=str(cache_dir), local_dir_use_symlinks=False)
    pt_candidates = list(Path(local_dir).glob("*.pt"))
    if not pt_candidates:
        raise FileNotFoundError("No .pt file found in the snapshot directory!")
    resolved_ckpt = str(pt_candidates[0])
    print(f"Resolved HF checkpoint: {resolved_ckpt}")

if resolved_ckpt is not None:
    CONFIG["CKPT_PATH"] = resolved_ckpt
    print("CONFIG['CKPT_PATH'] ->", CONFIG["CKPT_PATH"])
else:
    print("[WARN] No checkpoint chosen. Set one of the flags to True or manually provide CKPT_PATH.")

# %% [code]
class OutpaintDataset(Dataset):
    """
    返回：
      img:    [3,H,W] in [-1, 1]
      mask2d: [1,H,W] ∈ {0,1}，1 表示需要修补（被遮挡）区域
      bbox:   [4] 归一化 (x1,y1,x2,y2)
    """
    def __init__(self, root_dir: str, img_size: int = 512, masks_per_image: int = 80,
                 rotate=True, check_white=False):
        self.root_dir = root_dir
        self.img_files = [os.path.join(root_dir, f) for f in os.listdir(root_dir)
                          if f.lower().endswith((".png", ".jpg", ".jpeg"))]
        self.img_files.sort()
        self.img_size = img_size
        self.masks_per_image = masks_per_image
        tfs = [transforms.Resize((img_size, img_size))]
        if rotate:
            tfs.insert(0, transforms.RandomChoice([
                transforms.Lambda(lambda x: x),
                transforms.Lambda(lambda x: TF.rotate(x, 90)),
                transforms.Lambda(lambda x: TF.rotate(x, 180)),
                transforms.Lambda(lambda x: TF.rotate(x, 270)),
            ]))
        tfs += [
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),  # to [-1,1]
        ]
        self.transform = transforms.Compose(tfs)
        self.check_white = check_white

    def __len__(self):
        return max(1, len(self.img_files)) * self.masks_per_image

    @staticmethod
    def _gen_random_bbox() -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        # 带偏置的随机 bbox，覆盖极端/中等长宽比
        if random.random() < 0.5:
            aspect_ratio = random.uniform(1.0, 33.0)
        else:
            aspect_ratio = random.uniform(0.03, 1.0)
        area = random.uniform(0.1, 0.5) ** 2
        w = min(np.sqrt(area * aspect_ratio), 0.99)
        h = min(np.sqrt(area / aspect_ratio), 0.99)
        x1 = random.uniform(0.05, 1.0 - w - 0.01)
        y1 = random.uniform(0.05, 1.0 - h - 0.01)
        return (
            torch.tensor(x1, dtype=torch.float32),
            torch.tensor(y1, dtype=torch.float32),
            torch.tensor(x1 + w, dtype=torch.float32),
            torch.tensor(y1 + h, dtype=torch.float32),
        )

    def _check_mask_validity(self, img: torch.Tensor, mask2d: torch.Tensor) -> bool:
        # 避免大面积纯白（+1）的空洞作为训练区域（可选）
        sel = img[:, mask2d[0] == 1]
        if sel.numel() == 0:
            return False
        white = (sel == 1.0).all(dim=0)
        white_ratio = white.float().mean()
        return bool(white_ratio < 0.9)

    def __getitem__(self, idx):
        img_idx = idx // self.masks_per_image
        if len(self.img_files) == 0:
            raise RuntimeError(f"No images found in: {self.root_dir}")
        img = Image.open(self.img_files[img_idx % len(self.img_files)]).convert("RGB")
        img = self.transform(img)  # [3,512,512] in [-1,1]

        bbox = self._gen_random_bbox()
        _, H, W = img.shape
        x1 = int(bbox[0].item() * W); y1 = int(bbox[1].item() * H)
        x2 = int(bbox[2].item() * W); y2 = int(bbox[3].item() * H)

        mask2d = torch.zeros((1, H, W), dtype=torch.float32)
        mask2d[:, y1:y2, x1:x2] = 1.0

        # 若开启检查且不满足纯白比例阈值，则重新采样
        if self.check_white and not self._check_mask_validity(img, mask2d):
            return self.__getitem__(random.randint(0, len(self)-1))

        bbox_tensor = torch.stack(bbox)  # [4]
        return img, mask2d, bbox_tensor

# %% [code]
@torch.no_grad()
def save_image_grid(tensor_bchw: torch.Tensor, path: str, nrow: int = 4):
    # 保存 [-1,1] 到 PNG
    t = tensor_bchw.clamp(-1, 1)
    t = (t * 0.5 + 0.5).cpu().numpy()  # [B, C, H, W]
    B = t.shape[0]
    C, H, W = t.shape[1], t.shape[2], t.shape[3]
    rows = (B + nrow - 1) // nrow
    canvas = np.ones((rows * H, nrow * W, C), dtype=np.float32)
    for i in range(B):
        r = i // nrow; c = i % nrow
        canvas[r*H:(r+1)*H, c*W:(c+1)*W] = np.transpose(t[i], (1, 2, 0))
    Image.fromarray((canvas * 255).astype(np.uint8)).save(path)

# %% [code]
# DataLoader
train_set = OutpaintDataset(CONFIG["TRAIN_ROOT"], img_size=CONFIG["IMAGE_SIZE"])
val_set   = OutpaintDataset(CONFIG["VAL_ROOT"],   img_size=CONFIG["IMAGE_SIZE"])

train_loader = DataLoader(
    train_set, batch_size=CONFIG["BATCH"], shuffle=True,
    num_workers=CONFIG["NUM_WORKERS"], pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_set, batch_size=CONFIG["BATCH"], shuffle=False,
    num_workers=CONFIG["NUM_WORKERS"], pin_memory=True, drop_last=False
)

print("Train iters:", len(train_loader), " Val iters:", len(val_loader))

# %% [code]
# 创建模型并加载预训练权重

defaults = model_and_diffusion_defaults()
defaults.update(dict(
    image_size=CONFIG["IMAGE_SIZE"],
    num_channels=256,
    num_res_blocks=2,
    learn_sigma=True,
    class_cond=CONFIG["CLASS_COND"],
    use_scale_shift_norm=True,
    attention_resolutions="32,16,8",
    dropout=0.0,
    resblock_updown=True,
    use_fp16=True,
))
model, diffusion = create_model_and_diffusion(**defaults)
model.to(device)
model.convert_to_fp16()

assert os.path.isfile(CONFIG["CKPT_PATH"]), f"Checkpoint not found: {CONFIG['CKPT_PATH']}"
state = torch.load(CONFIG["CKPT_PATH"], map_location="cpu")
missing, unexpected = model.load_state_dict(state, strict=False)
print("[load] missing:", missing)
print("[load] unexpected:", unexpected)

opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"])

# %% [code]
# 训练/验证/采样工具函数

def masked_mse(pred: torch.Tensor, target: torch.Tensor, mask2d: torch.Tensor) -> torch.Tensor:
    m = mask2d
    num = (m.sum(dim=(1,2,3)) + 1e-8)
    err = ((pred - target) ** 2) * m
    err = err.sum(dim=(1,2,3)) / num
    return err.mean()

def q_sample(x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
    return diffusion.q_sample(x_start=x0, t=t, noise=noise)

def train_step(x0: torch.Tensor, mask2d: torch.Tensor, labels=None) -> torch.Tensor:
    B = x0.shape[0]
    t = torch.randint(low=0, high=diffusion.num_timesteps, size=(B,), device=device)
    noise = torch.randn_like(x0)

    # 仅在 mask 内加噪
    x_t_full = q_sample(x0, t, noise)
    x_in = x_t_full * mask2d + x0 * (1.0 - mask2d)

    model_kwargs = {}
    if labels is not None:
        model_kwargs["y"] = labels

    eps_pred = model(x_in, t, **model_kwargs)
    if eps_pred.shape[1] != x0.shape[1]:
        eps_pred = eps_pred[:, :x0.shape[1]]  # learn_sigma=True 时，前 3 通道是 eps

    loss = masked_mse(eps_pred, noise, mask2d)
    return loss

@torch.no_grad()
def val_epoch() -> float:
    model.eval()
    losses = []
    for x0, mask2d, _ in tqdm(val_loader, desc="Validating"):
        x0 = x0.to(device); mask2d = mask2d.to(device)
        loss = train_step(x0, mask2d)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses)) if losses else 0.0

@torch.no_grad()
def sample_inpaint(x0: torch.Tensor, mask2d: torch.Tensor, steps: int = None) -> torch.Tensor:
    model.eval()
    B = x0.shape[0]
    steps = steps or diffusion.num_timesteps

    # 从噪声开始（mask 内），mask 外保持为原图，并在每步后重新强制覆盖
    x = torch.randn_like(x0)
    x = x * mask2d + x0 * (1.0 - mask2d)

    for i in reversed(range(steps)):
        t = torch.full((B,), i, device=device, dtype=torch.long)
        eps_pred = model(x, t)
        if eps_pred.shape[1] != x0.shape[1]:
            eps_pred = eps_pred[:, :x0.shape[1]]

        out = diffusion.p_mean_variance(model, x, t, model_kwargs={})
        mean, var, log_var = out["mean"], out["variance"], out["log_variance"]
        if i > 0:
            noise = torch.randn_like(x)
            x = mean + torch.exp(0.5 * log_var) * noise
        else:
            x = mean

        # 重新覆盖上下文
        x = x * mask2d + x0 * (1.0 - mask2d)

    model.train()
    return x

# %% [code]
# 训练主循环
best_val = float('inf')
bad = 0
train_losses, val_losses = [], []

for epoch in range(1, CONFIG["EPOCHS"] + 1):
    epoch_losses = []
    for x0, mask2d, _ in tqdm(train_loader, desc=f"Epoch {epoch} [train]"):
        x0 = x0.to(device); mask2d = mask2d.to(device)

        loss = train_step(x0, mask2d)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        epoch_losses.append(loss.item())

    tr = float(np.mean(epoch_losses)) if epoch_losses else 0.0
    va = val_epoch()
    train_losses.append(tr); val_losses.append(va)
    print(f"Epoch {epoch}: train={tr:.6f} val={va:.6f}")

    # 保存 best
    if va < best_val:
        best_val = va; bad = 0
        torch.save(model.state_dict(), os.path.join(CONFIG["OUTDIR"], "model_best.pt"))
        torch.save(opt.state_dict(),   os.path.join(CONFIG["OUTDIR"], "opt_best.pt"))
    else:
        bad += 1
        print(f"Early stopping patience {bad}/{CONFIG['PATIENCE']}")

    # 采样预览
    if epoch % CONFIG["SAMPLE_EVERY"] == 0:
        try:
            x0_vis, mask_vis, _ = next(iter(val_loader))
        except StopIteration:
            x0_vis, mask_vis, _ = next(iter(train_loader))
        x0_vis = x0_vis[:4].to(device); mask_vis = mask_vis[:4].to(device)
        with torch.no_grad():
            out = sample_inpaint(x0_vis, mask_vis, steps=CONFIG["SAMPLE_STEPS"])
        grid = torch.cat([x0_vis * (1 - mask_vis) + mask_vis * (-1), x0_vis, out], dim=0)
        save_image_grid(grid, os.path.join(CONFIG["OUTDIR"], f"epoch_{epoch:04d}.png"), nrow=4)

    if bad >= CONFIG["PATIENCE"]:
        print("Stopping early.")
        break

# 可视化 Loss 曲线
plt.figure(figsize=(7,4))
plt.plot(train_losses, label="train"); plt.plot(val_losses, label="val")
plt.xlabel("Epoch"); plt.ylabel("Masked MSE (eps)"); plt.grid(True); plt.legend(); plt.tight_layout()
plt.show()

# %% [code]
# Ad-hoc 采样 Demo（自行给出 x0.png 与 mask.png（1 表示需要修补区域），均为 512x512）
from torchvision import transforms as _T

def load_rgb(path, image_size=CONFIG["IMAGE_SIZE"]):
    im = Image.open(path).convert("RGB").resize((image_size, image_size))
    t = _T.ToTensor()(im) * 2 - 1
    return t.unsqueeze(0).to(device)

def load_mask(path, image_size=CONFIG["IMAGE_SIZE"]):
    im = Image.open(path).convert("L").resize((image_size, image_size))
    a = np.array(im) / 255.0
    m = (a >= 0.5).astype(np.float32)
    t = torch.from_numpy(m)[None, None, ...]
    return t.to(device)

# 示例：
# img0 = load_rgb("/path/to/x0.png")
# m2d  = load_mask("/path/to/mask.png")
# with torch.no_grad():
#     out = sample_inpaint(img0, m2d, steps=CONFIG["SAMPLE_STEPS"])
# save_image_grid(torch.cat([img0 * (1 - m2d) + m2d*(-1), img0, out], dim=0),
#                 os.path.join(CONFIG["OUTDIR"], "ad_hoc_inpaint.png"), nrow=3)
